# Andre Pallante | Cardinals Player Valuation

A guided analysis of a 2027 tender decision. Saved outputs show the results; the source CSVs are not included in this portfolio.

## 0. Decision frame

**Player:** Andre Pallante

**Decision:** Tender or non-tender for the 2027 season.

**Primary question:** Is the selected player's expected 2027 value greater than his projected salary and the opportunity cost of retaining his roster spot?

**Unit of analysis:** One player-season decision.

**Decision horizon:** 2027, while considering 2028 roster flexibility and internal pitching development.

## 1. Imports and notebook settings

Use the smallest practical dependency set. The initial workflow should only require standard Python plus `pandas`; add other packages only when there is a clear need.

In [5]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 160)

PLAYER_MLBAM_ID = 669467

## 2. Source files and provenance

The main public sources for this analysis are:

- `data/fangraphs_2026_stl_pitchers_actuals_clean.csv` — 2026 Cardinals pitcher outcomes.
- `data/fangraphs_2027_stl_pitchers_projections_zips_clean.csv` — 2027 ZiPS projection.
- `data/fangraphs_2026_stl_pitchers_projections_zips_clean.csv` — 2026 ZiPS projection, useful for retrospective comparison.
- `data/fangraphs_2026_stl_pitchers_projections_zips_dc_clean.csv` — 2026 ZiPS DC projection, useful for retrospective comparison.
- `data/player_index.csv` — player identity and file coverage map.
- `data/contract_research.csv` — contract, arbitration, and salary research.
- `data/service_time_reconciliation_2026_with_opening_status.csv` — service-time and roster-status context.
- `data/player_evaluation.csv` — decision-workspace fields.
- `data/player_universe.csv` — broader Cardinals player universe.
- `data/projection_inputs.csv` — projection-source notes and inputs, where applicable.
- `data/raw/baseballsavant_2026_MLB_pitchers_summary.csv` — optional Statcast context.

In [6]:
# Find the project directory without depending on the notebook's launch directory.
project_candidates = [Path.cwd(), Path.cwd().parent, Path.cwd().parent.parent]
PROJECT_DIR = next(
    path for path in project_candidates
    if (path / "data").is_dir() and (path / "README.md").is_file()
)
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"

source_files = {
    "pitcher_actuals_2026": "data/fangraphs_2026_stl_pitchers_actuals_clean.csv",
    "hitter_actuals_2026": "data/fangraphs_2026_stl_hitters_actuals_clean.csv",
    "pitcher_zips_2026": "data/fangraphs_2026_stl_pitchers_projections_zips_clean.csv",
    "hitter_zips_2026": "data/fangraphs_2026_stl_hitters_projections_zips_clean.csv",
    "pitcher_zips_dc_2026": "data/fangraphs_2026_stl_pitchers_projections_zips_dc_clean.csv",
    "hitter_zips_dc_2026": "data/fangraphs_2026_stl_hitters_projections_zips_dc_clean.csv",
    "pitcher_zips_2027": "data/fangraphs_2027_stl_pitchers_projections_zips_clean.csv",
    "hitter_zips_2027": "data/fangraphs_2027_stl_hitters_projections_zips_clean.csv",
    "player_index": "data/player_index.csv",
    "contract_research": "data/contract_research.csv",
    "player_evaluation": "data/player_evaluation.csv",
    "player_universe": "data/player_universe.csv",
    "projection_inputs": "data/projection_inputs.csv",
    "service_time": "data/service_time_reconciliation_2026_with_opening_status.csv",
    "roster_snapshot": "data/api_roster_snapshot_2026-10-12.csv",
    "opening_status": "data/opening_status_2026-03-26.csv",
    "savant_pitchers_2026": "data/raw/baseballsavant_2026_MLB_pitchers_summary.csv",
    "savant_hitters_2026": "data/raw/baseballsavant_2026_MLB_hitters_summary.csv",
}

def read_source_csv(relative_path):
    """Read a source CSV without changing the saved source file."""
    return pd.read_csv(PROJECT_DIR / relative_path, encoding="utf-8-sig")

tables = {name: read_source_csv(path) for name, path in source_files.items()}
table_shapes = pd.DataFrame(
    [
        {"table": name, "rows": frame.shape[0], "columns": frame.shape[1]}
        for name, frame in tables.items()
    ]
).sort_values("table")
table_shapes

,table,rows,columns
9,contract_research,40,14
1,hitter_actuals_2026,22,66
3,hitter_zips_2026,42,45
7,hitter_zips_2027,42,45
5,hitter_zips_dc_2026,16,45
15,opening_status,41,9
0,pitcher_actuals_2026,30,61
2,pitcher_zips_2026,36,40
6,pitcher_zips_2027,36,40
4,pitcher_zips_dc_2026,21,40


## 3. Data coverage and schema checks

Before joining or interpreting anything, verify:

- row counts for every source;
- column names and data types;
- whether `MLBAMID` is present and unique where expected;
- whether players name appears in each source;
- whether projection rows represent the same player and season;
- missing values in fields used for the decision; and
- whether any records are duplicated or unexpectedly absent.

Use `MLBAMID` as the primary join key. Use player names for display and manual review, not as the primary identity key.

In [7]:
required_columns = {
    "pitcher_actuals_2026": {"Name", "Team", "Season", "IP", "WAR", "MLBAMID"},
    "pitcher_zips_2027": {"Name", "Team", "IP", "WAR", "MLBAMID"},
    "pitcher_zips_2026": {"Name", "Team", "IP", "WAR", "MLBAMID"},
    "pitcher_zips_dc_2026": {"Name", "Team", "IP", "WAR", "MLBAMID"},
    "player_index": {"MLBAMID", "Name", "Role", "ProjectionSeason", "SourceFile"},
    "contract_research": {"mlb_id", "player_name", "arbitration_status"},
    "service_time": {"mlb_id", "player_name", "service_time_2026"},
}

id_candidates = ["MLBAMID", "mlb_id", "player_id", "PlayerId"]

def find_id_column(frame):
    return next((column for column in id_candidates if column in frame.columns), None)

qc_rows = []
for name, frame in tables.items():
    id_column = find_id_column(frame)
    duplicate_ids = None
    selected_player_rows = None
    if id_column is not None:
        ids = frame[id_column].astype("string")
        duplicate_ids = int(ids.dropna().duplicated().sum())
        selected_player_rows = int((ids == str(PLAYER_MLBAM_ID)).sum())
    missing_required = sorted(required_columns.get(name, set()) - set(frame.columns))
    qc_rows.append(
        {
            "table": name,
            "id_column": id_column,
            "duplicate_id_rows": duplicate_ids,
            "selected_player_rows": selected_player_rows,
            "missing_required_columns": ", ".join(missing_required),
        }
    )

schema_qc = pd.DataFrame(qc_rows).sort_values("table")
schema_qc

,table,id_column,duplicate_id_rows,selected_player_rows,missing_required_columns
9,contract_research,mlb_id,0.0,1.0,
1,hitter_actuals_2026,MLBAMID,0.0,0.0,
3,hitter_zips_2026,MLBAMID,0.0,0.0,
7,hitter_zips_2027,MLBAMID,0.0,0.0,
5,hitter_zips_dc_2026,MLBAMID,0.0,0.0,
15,opening_status,mlb_id,0.0,1.0,
0,pitcher_actuals_2026,MLBAMID,0.0,1.0,
2,pitcher_zips_2026,MLBAMID,0.0,1.0,
6,pitcher_zips_2027,MLBAMID,0.0,1.0,
4,pitcher_zips_dc_2026,MLBAMID,0.0,1.0,


## 4. Player identity and joins

Build the players analysis row by joining the relevant tables on `MLBAMID`. Keep the source-specific columns distinguishable when names overlap.

Expected join inputs:

1. `player_index.csv` for identity and coverage.
2. 2026 actuals for descriptive performance.
3. 2027 ZiPS for the primary public projection.
4. 2026 ZiPS and ZiPS DC for retrospective projection context.
5. Contract research for salary and arbitration context.
6. Service-time/roster data for eligibility and roster context.

Do not force a join when a source lacks player. Record the missingness and explain its impact.

In [8]:
def filter_to_player(frame, player_mlbam_id=PLAYER_MLBAM_ID):
    id_column = find_id_column(frame)
    if id_column is None:
        return frame.iloc[0:0].copy()
    ids = frame[id_column].astype("string")
    return frame.loc[ids == str(player_mlbam_id)].copy()

selected_player_source_rows = {
    name: filter_to_player(frame)
    for name, frame in tables.items()
}

selected_player_coverage = pd.DataFrame(
    [
        {
            "table": name,
            "rows_for_player": frame.shape[0],
            "columns": frame.shape[1],
        }
        for name, frame in selected_player_source_rows.items()
    ]
).sort_values("table")
selected_player_coverage

,table,rows_for_player,columns
9,contract_research,1,14
1,hitter_actuals_2026,0,66
3,hitter_zips_2026,0,45
7,hitter_zips_2027,0,45
5,hitter_zips_dc_2026,0,45
15,opening_status,1,9
0,pitcher_actuals_2026,1,61
2,pitcher_zips_2026,1,40
6,pitcher_zips_2027,1,40
4,pitcher_zips_dc_2026,1,40


## 5. 2026 actual performance

Describe what happened in 2026 before discussing what is expected in 2027.

In [9]:
actuals = tables["pitcher_actuals_2026"]

selected_player_actuals = actuals.loc[
    actuals["MLBAMID"] == PLAYER_MLBAM_ID
].copy()

actual_columns = [
    "Name",
    "Team",
    "Season",
    "Age",
    "G",
    "GS",
    "IP",
    "ERA",
    "FIP",
    "WAR",
    "K/9",
    "BB/9",
    "HR/9",
    "WHIP",
    "BABIP",
    "SIERA",
    "K-BB%",
    "Stuff+",
    "Location+",
    "Pitching+",
]

selected_player_actuals.loc[:, actual_columns]

,Name,Team,Season,Age,G,GS,IP,ERA,FIP,WAR,K/9,BB/9,HR/9,WHIP,BABIP,SIERA,K-BB%,Stuff+,Location+,Pitching+
19,Andre Pallante,STL,2026,27,29,29,162.0,3.833333,3.741651,2.69539,6.111111,2.777778,0.666667,1.271605,0.287425,4.294737,0.088365,97.036597,104.763107,103.042355


- Age 27; 29 starts; 162 IP.
- 3.83 ERA, 3.74 FIP, 2.69 FanGraphs WAR: durable, useful mid-rotation production.
- 6.11 K/9, 2.78 BB/9, 0.67 HR/9, 1.27 WHIP, .287 BABIP.
- K-BB% was 8.8%
- FIP was close to ERA, meaning his results were broadly supported by strikeouts, walks, and home-run outcomes
- SIERA was 4.29, higher than both ERA and FIP. Because SIERA incorporates strikeouts, walks, home runs, and batted-ball data, this raises some regression concern
- Stuff+ was 97, Location+ was 105, and Pitching+ was 103. Relative to a 100 average:- Slightly below-average stuff.
  - Above-average location.
  - Slightly above-average overall pitching process.

Overall: Pallante had a valuable and durable 2026, but his low strikeout rate, modest K-BB%, and higher SIERA suggest we should not automatically assume he will repeat a 3.83 ERA.

## 6. Projection comparison

Compare ZiPS and ZiPS DC when both are available. Do not treat disagreement as an error; treat it as evidence of projection uncertainty or different playing-time assumptions.

In [10]:
comparison_sources ={
    "2026 Actual": tables["pitcher_actuals_2026"],
    "2026 ZiPS": tables["pitcher_zips_2026"],
    "2026 ZiPS DC": tables["pitcher_zips_dc_2026"],
    "2027 ZiPS": tables["pitcher_zips_2027"],
}

comparison_columns = [
    "IP",
    "GS",
    "ERA",
    "FIP",
    "WAR",
    "K/9",
    "BB/9",
    "K/BB",
    "K-BB%",
    "HR/9",
]

comparison_rows = []

for source_name, frame, in comparison_sources.items():
    required_columns = ["MLBAMID"] + comparison_columns
    missing_columns = sorted(set(required_columns) - set(frame.columns))

    if missing_columns:
        raise KeyError(
            f"{source_name} is missing these columns: {missing_columns}"
        )
    player_row = frame.loc[
        frame["MLBAMID"].eq(PLAYER_MLBAM_ID),
        required_columns,
    ].copy()

    if len(player_row) != 1:
        raise ValueError(
            f"{source_name} returned {len(player_row)} rows for {PLAYER_NAME}; expected 1."
        )

    player_row.insert(0, "Source", source_name)
    comparison_rows.append(player_row)

projection_comparison = pd.concat(
    comparison_rows,
    ignore_index=True,
).drop(columns="MLBAMID")

projection_comparison

,Source,IP,GS,ERA,FIP,WAR,K/9,BB/9,K/BB,K-BB%,HR/9
0,2026 Actual,162.0,29,3.833333,3.741651,2.69539,6.111111,2.777778,2.20000,0.088365,0.666667
1,2026 ZiPS,135.7,24,4.244660,4.093130,1.45872,6.433310,3.448780,1.86538,0.076531,0.795873
2,2026 ZiPS DC,129.0,24,4.244660,4.093130,1.46760,6.433310,3.448780,1.86538,0.076531,0.795873
3,2027 ZiPS,126.7,23,4.191000,4.050680,1.43049,6.393050,3.338600,1.91489,0.078611,0.781373


- Pallante outperformed both 2026 ZiPS projections in workload, ERA, FIP, WAR, BB/9, K/BB, K-BB%, and HR/9.
- His K/9 was lower than projected, so the improvement came primarily from throwing more innings, limiting walks and home runs, and producing better overall run prevention.
- ZiPS and ZiPS DC were similar in projected performance rates; most of their difference came from playing-time assumptions.
- The 2027 ZiPS projection remains similar on a rate basis but projects fewer innings and starts, resulting in only 1.43 WAR.
T- he 2026 results support viewing Pallante as a useful starter, but his low strikeout rate and 4.29 SIERA create some regression risk.
- I would not automatically increase his 2027 projection because he outperformed once. His 2026 performance supports a provisional tender evaluation, which now needs to be compared against his projected arbitration salary and internal alternatives.

## 7. Organization-wide 2027 ZiPS pitcher comparison

Compare the selected players 2027 ZiPS projection with every pitcher included in the 2027 ZiPS export.

The table includes roster and contract context when available. Projection-only players remain in the table but should not automatically be treated as realistic internal replacements.

Gap calculations use:

candidate projection − baseline player projection

For metrics where lower is better, a negative gap favors the candidate.

In [7]:
roster = tables["player_index"].copy()
zips_2027 = tables["pitcher_zips_2027"].copy()
contracts = tables["contract_research"].copy()

# Remove hidden BOM characters and standardize ID fields.
for frame in [roster, zips_2027]:
    frame.columns = (
        frame.columns
        .str.replace("\ufeff", "", regex=False)
        .str.strip()
    )
    frame["MLBAMID"] = frame["MLBAMID"].astype("string").str.strip()

contracts = contracts.rename(columns={"mlb_id": "MLBAMID"})
contracts["MLBAMID"] = (
    contracts["MLBAMID"]
    .astype("string")
    .str.strip()
)

# These are the projection metrics used in the comparison.
projection_metric_map = {
    "G": "G",
    "GS": "GS",
    "IP": "IP",
    "TBF": "TBF",
    "QS": "QS",
    "ERA": "ERA",
    "FIP": "FIP",
    "K/9": "K9",
    "BB/9": "BB9",
    "K/BB": "KBB",
    "K-BB%": "KBB_pct",
    "K%": "K_pct",
    "BB%": "BB_pct",
    "HR/9": "HR9",
    "WHIP": "WHIP",
    "BABIP": "BABIP",
    "LOB%": "LOB_pct",
    "GB%": "GB_pct",
    "HR/FB": "HR_FB",
    "AVG": "AVG",
    "WAR": "WAR",
    "RA9-WAR": "RA9_WAR",
}

source_projection_columns = [
    "MLBAMID",
    "Name",
    "Team",
] + list(projection_metric_map.keys())

zips_2027_selected = zips_2027[
    source_projection_columns
].rename(
    columns={
        source_name: f"ZIPS_2027_{output_name}"
        for source_name, output_name in projection_metric_map.items()
    }
)

# Keep organizational context separate from projection statistics.
roster_context = roster[
    [
        "MLBAMID",
        "Role",
        "roster_group",
        "projection_status",
        "ProjectionSeason",
        "ProjectionModel",
    ]
].copy()

contract_context = contracts[
    [
        "MLBAMID",
        "contract_bucket",
        "arbitration_status",
        "service_time_total",
        "service_time_2026",
        "MiLB_options_remaining",
        "contract_options_status",
        "confidence",
        "notes",
    ]
].rename(
    columns={
        "confidence": "contract_confidence",
        "notes": "contract_notes",
    }
)

pitcher_comparison = (
    zips_2027_selected
    .merge(
        roster_context,
        on="MLBAMID",
        how="left",
        validate="many_to_one",
    )
    .merge(
        contract_context,
        on="MLBAMID",
        how="left",
        validate="many_to_one",
    )
)

numeric_projection_columns = [
    f"ZIPS_2027_{output_name}"
    for output_name in projection_metric_map.values()
]

pitcher_comparison[numeric_projection_columns] = (
    pitcher_comparison[numeric_projection_columns]
    .apply(pd.to_numeric, errors="coerce")
)

In [8]:
baseline_player_id = str(PLAYER_MLBAM_ID)

baseline_player_row = pitcher_comparison.loc[
    pitcher_comparison["MLBAMID"].eq(baseline_player_id)
]

if len(baseline_player_row) != 1:
    raise ValueError("Expected exactly one baseline player row.")

baseline_player_row = baseline_player_row.iloc[0]

for output_name in projection_metric_map.values():
    projection_column = f"ZIPS_2027_{output_name}"
    gap_column = f"Gap_{output_name}_vs_Baseline"

    pitcher_comparison[gap_column] = (
        pitcher_comparison[projection_column]
        - baseline_player_row[projection_column]
    )

In [9]:
gap_output_names = [
    "WAR",
    "RA9_WAR",
    "IP",
    "GS",
    "G",
    "TBF",
    "QS",
    "ERA",
    "FIP",
    "KBB_pct",
    "K_pct",
    "BB_pct",
    "K9",
    "BB9",
    "KBB",
    "HR9",
    "WHIP",
    "BABIP",
    "LOB_pct",
    "GB_pct",
    "HR_FB",
    "AVG",
]

gap_columns = [
    f"Gap_{output_name}_vs_Baseline"
    for output_name in gap_output_names
]

projection_columns = [
    f"ZIPS_2027_{output_name}"
    for output_name in gap_output_names
]

context_columns = [
    "roster_group",
    "contract_bucket",
    "arbitration_status",
    "service_time_total",
    "service_time_2026",
    "MiLB_options_remaining",
    "contract_options_status",
    "MLBAMID",
]

pitcher_comparison_table = (
    pitcher_comparison[
        ["Name"] + gap_columns + projection_columns + context_columns
    ]
    .sort_values(
        ["Gap_WAR_vs_Baseline", "Gap_IP_vs_Baseline"],
        ascending=[False, False],
    )
    .reset_index(drop=True)
)

pitcher_comparison_table

,Name,Gap_WAR_vs_Baseline,Gap_RA9_WAR_vs_Baseline,Gap_IP_vs_Baseline,Gap_GS_vs_Baseline,Gap_G_vs_Baseline,Gap_TBF_vs_Baseline,Gap_QS_vs_Baseline,Gap_ERA_vs_Baseline,Gap_FIP_vs_Baseline,Gap_KBB_pct_vs_Baseline,Gap_K_pct_vs_Baseline,Gap_BB_pct_vs_Baseline,Gap_K9_vs_Baseline,Gap_BB9_vs_Baseline,Gap_KBB_vs_Baseline,Gap_HR9_vs_Baseline,Gap_WHIP_vs_Baseline,Gap_BABIP_vs_Baseline,Gap_LOB_pct_vs_Baseline,Gap_GB_pct_vs_Baseline,Gap_HR_FB_vs_Baseline,Gap_AVG_vs_Baseline,ZIPS_2027_WAR,ZIPS_2027_RA9_WAR,ZIPS_2027_IP,ZIPS_2027_GS,ZIPS_2027_G,ZIPS_2027_TBF,ZIPS_2027_QS,ZIPS_2027_ERA,ZIPS_2027_FIP,ZIPS_2027_KBB_pct,ZIPS_2027_K_pct,ZIPS_2027_BB_pct,ZIPS_2027_K9,ZIPS_2027_BB9,ZIPS_2027_KBB,ZIPS_2027_HR9,ZIPS_2027_WHIP,ZIPS_2027_BABIP,ZIPS_2027_LOB_pct,ZIPS_2027_GB_pct,ZIPS_2027_HR_FB,ZIPS_2027_AVG,roster_group,contract_bucket,arbitration_status,service_time_total,service_time_2026,MiLB_options_remaining,contract_options_status,MLBAMID
0,Michael McGreevy,0.683820,0.972950,22.3,4,-2,73,NaN,-0.32523,-0.22427,0.040744,0.009660,-0.031085,0.13044,-1.28491,1.26158,0.185070,-0.14277,-0.003205,0.016443,NaN,NaN,0.000128,2.114310,2.424520,149.0,27,28,620,NaN,3.86577,3.82641,0.119355,0.174194,0.054839,6.52349,2.05369,3.17647,0.966443,1.21477,0.285403,0.712484,NaN,NaN,0.252144,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 ARB 1; 2030 A...,1.091,1.000,1.0,NOT SHOWN,700241
1,Matthew Liberatore,0.409970,0.391040,-0.4,-1,-1,-16,NaN,-0.27176,-0.27421,0.055099,0.036973,-0.018127,1.23165,-0.77328,1.05733,0.144993,-0.13030,-0.001345,0.005713,NaN,NaN,-0.008662,1.840460,1.842610,126.3,22,29,531,NaN,3.91924,3.77647,0.133710,0.201507,0.067797,7.62470,2.56532,2.97222,0.926366,1.22724,0.287263,0.701754,NaN,NaN,0.243354,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 ARB 4; 2030 FREE ...,3.144,1.000,1.0,NOT SHOWN,669461
2,Andre Pallante,0.000000,0.000000,0.0,0,0,0,NaN,0.00000,0.00000,0.000000,0.000000,0.000000,0.00000,0.00000,0.00000,0.000000,0.00000,0.000000,0.000000,NaN,NaN,0.000000,1.430490,1.451570,126.7,23,30,547,NaN,4.19100,4.05068,0.078611,0.164534,0.085923,6.39305,3.33860,1.91489,0.781373,1.35754,0.288608,0.696041,NaN,NaN,0.252016,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 FREE AGENT,4.145,1.000,1.0,NOT SHOWN,669467
3,Tekoah Roby,-0.062600,0.059020,-28.0,-2,-9,-133,NaN,-0.27003,-0.09487,0.059071,0.038365,-0.020706,1.26652,-0.87659,1.19622,0.312847,-0.14173,-0.006378,0.021013,NaN,NaN,-0.009196,1.367890,1.510590,98.7,21,21,414,NaN,3.92097,3.95581,0.137681,0.202899,0.065217,7.65957,2.46201,3.11111,1.094220,1.21581,0.282230,0.717054,NaN,NaN,0.242820,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,0.000,0.000,1.0,NOT SHOWN,694358
4,Daniel Eagen,-0.148640,0.489350,-17.4,0,-7,-81,NaN,-0.32092,0.02682,0.028686,0.037183,0.008497,1.34711,0.28446,0.22147,0.206733,-0.03092,-0.006068,0.044989,NaN,NaN,-0.012111,1.281850,1.940920,109.3,23,23,466,NaN,3.87008,4.07750,0.107296,0.201717,0.094421,7.74016,3.62306,2.13636,0.988106,1.32662,0.282540,0.741030,NaN,NaN,0.239905,projection_export_only,NaN,NaN,NaN,NaN,NaN,NaN,811666
5,Tink Hence,-0.203390,-0.286300,-35.0,3,-4,-155,NaN,-0.06886,-0.05404,0.041287,0.049752,0.008465,1.85123,0.29281,0.35538,0.200088,-0.04892,-0.006755,0.011506,NaN,NaN,-0.016889,1.227100,1.165270,91.7,26,26,392,NaN,4.12214,3.99664,0.119898,0.214286,0.094388,8.24428,3.63141,2.27027,0.981461,1.30862,0.281853,0.707547,NaN,NaN,0.235127,projection_export_only,NaN,NaN,NaN,NaN,NaN,NaN,693311
6,Quinn Mathews,-0.303110,-0.130540,-24.4,1,-6,-100,NaN,-0.05610,0.14995,0.035483,0.061417,0.025934,2.49258,1.06023,0.10511,0.186369,0.01098,-0.005454,0.026270,NaN,NaN,-0.021837,1.127380,1.321030,102.3,24,24,447,NaN,4.13490,4.20063,0.114094,0.225951,0.111857,8.88563,4.39883,2.02000,0.967742,1.36852,0.283154,0.722311,NaN,NaN,0.230179,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 PRE-ARB; 2030...,0.041,0.041,3.0,NOT SHOWN,687273
7,Sem Robberse,-0.309590,-0.247320,-27.7,-4,-10,-121,NaN,-0.00918,0.11476,0.022328,0.013870,-0.008458

### Decision-oriented comparison view

This narrower table highlights the metrics most relevant to evaluating players projected value and potential internal alternatives. The full master table remains available above for reference.

In [10]:
decision_columns = [
    "Name",
    "Gap_WAR_vs_Baseline",
    "Gap_RA9_WAR_vs_Baseline",
    "Gap_IP_vs_Baseline",
    "Gap_GS_vs_Baseline",
    "Gap_FIP_vs_Baseline",
    "Gap_KBB_pct_vs_Baseline",
    "ZIPS_2027_WAR",
    "ZIPS_2027_RA9_WAR",
    "ZIPS_2027_IP",
    "ZIPS_2027_GS",
    "ZIPS_2027_FIP",
    "ZIPS_2027_KBB_pct",
    "roster_group",
    "contract_bucket",
    "arbitration_status",
    "MiLB_options_remaining",
    "contract_options_status",
]

decision_view = (
    pitcher_comparison[decision_columns]
    .sort_values(
        ["Gap_WAR_vs_Baseline", "Gap_IP_vs_Baseline"],
        ascending=[False, False],
        na_position="last",
    )
    .reset_index(drop=True)
)

decision_view

,Name,Gap_WAR_vs_Baseline,Gap_RA9_WAR_vs_Baseline,Gap_IP_vs_Baseline,Gap_GS_vs_Baseline,Gap_FIP_vs_Baseline,Gap_KBB_pct_vs_Baseline,ZIPS_2027_WAR,ZIPS_2027_RA9_WAR,ZIPS_2027_IP,ZIPS_2027_GS,ZIPS_2027_FIP,ZIPS_2027_KBB_pct,roster_group,contract_bucket,arbitration_status,MiLB_options_remaining,contract_options_status
0,Michael McGreevy,0.683820,0.972950,22.3,4,-0.22427,0.040744,2.114310,2.424520,149.0,27,3.82641,0.119355,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 ARB 1; 2030 A...,1.0,NOT SHOWN
1,Matthew Liberatore,0.409970,0.391040,-0.4,-1,-0.27421,0.055099,1.840460,1.842610,126.3,22,3.77647,0.133710,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 ARB 4; 2030 FREE ...,1.0,NOT SHOWN
2,Andre Pallante,0.000000,0.000000,0.0,0,0.00000,0.000000,1.430490,1.451570,126.7,23,4.05068,0.078611,40man_2026-10-12,ARBITRATION,2027 ARB 2; 2028 ARB 3; 2029 FREE AGENT,1.0,NOT SHOWN
3,Tekoah Roby,-0.062600,0.059020,-28.0,-2,-0.09487,0.059071,1.367890,1.510590,98.7,21,3.95581,0.137681,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,1.0,NOT SHOWN
4,Daniel Eagen,-0.148640,0.489350,-17.4,0,0.02682,0.028686,1.281850,1.940920,109.3,23,4.07750,0.107296,projection_export_only,NaN,NaN,NaN,NaN
5,Tink Hence,-0.203390,-0.286300,-35.0,3,-0.05404,0.041287,1.227100,1.165270,91.7,26,3.99664,0.119898,projection_export_only,NaN,NaN,NaN,NaN
6,Quinn Mathews,-0.303110,-0.130540,-24.4,1,0.14995,0.035483,1.127380,1.321030,102.3,24,4.20063,0.114094,40man_2026-10-12,PRE_ARBITRATION,2027 PRE-ARB; 2028 PRE-ARB; 2029 PRE-ARB; 2030...,3.0,NOT SHOWN
7,Sem Robberse,-0.309590,-0.247320,-27.7,-4,0.11476,0.022328,1.120900,1.204250,99.0,19,4.16544,0.100939,projection_export_only,NaN,NaN,NaN,NaN
8,Ixan Henderson,-0.347520,-0.252430,-18.0,-2,0.23441,0.014808,1.082970,1.199140,108.7,21,4.28509,0.093418,projection_export_only,NaN,NaN,NaN,NaN
9,Brycen Mautz,-0.439118,-0.456613,-21.4,1,0.29053,0.030799,0.991372,0.994957,105.3,24,4.34121,0.109409,40man_2026-10-12,PRE_ARBITRATION,PRE-ARB,2.0,NOT SHOWN


## 8. Salary, arbitration, and opportunity cost

Compare expected player value with expected cost.

### Salary assumptions

- 2026 Salary: $4.0 million
- 2027 status: Arbitration Year 2
- Just Baseball projects a $7.5 million 2027

## 9. Player interpretation

**What the evidence says:**

- Age 27; 29 starts; 162 IP; 3.83 ERA, 3.74 FIP, and 2.69 FanGraphs WAR in 2026.
- Pallante outperformed both 2026 ZiPS projections in workload and most outcome metrics. His K/9 was lower than projected, but his BB/9, K/BB, K-BB%, and HR/9 were better.
- The 2027 ZiPS projection is 126.7 IP, 23 GS, 4.19 ERA, 4.05 FIP, and 1.43 WAR. That is the third-highest projected WAR among pitchers in the 2027 ZiPS export.
- His 6.11 K/9, 8.8% K-BB%, 4.29 SIERA, and 97 Stuff+ create more regression concern than his 2026 ERA alone suggests.

**What the evidence does not establish:**

- His 2026 outperformance does not guarantee that he will repeat a 3.83 ERA in 2027.
- The public data do not provide the Cardinals' internal projection, a player-specific injury probability, or a precise arbitration outcome.

**Roster and organizational context:**

- There is no obvious one-for-one internal replacement for Pallante's projected bulk and reliability.
- Tekoah Roby is the most direct potential rotation replacement, but his post-Tommy John workload creates uncertainty. Daniel Eagen has projected value but is less likely to replace Pallante immediately from Double-A.
- Michael McGreevy and Matthew Liberatore project for more 2027 WAR, but they may already be needed for other rotation roles rather than serving as free replacements.
- Pallante also retains value as a starter, swingman, or long-relief option.

**Salary interpretation:**

- Just Baseball projects a $7.5 million 2027 salary. My working estimate is closer to approximately $6 million because of Pallante's low-strikeout profile, regression indicators, and the club's internal pitching depth. This is a baseball judgment, not the output of a formal salary model.

**Recommendation:**

- Tender Pallante. His projected value, durability, and lack of an equivalent internal replacement outweigh the risk that his 2026 ERA partially regresses. I would tender him, but I would be hesitant to value him as high as the $7.5 million public estimate.